<a href="https://colab.research.google.com/github/busybee-123/Pollinator_Cam/blob/main/Downloading_images_from_GBIF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download image dataset from taxa information

### Import packages and connect google drive

In [ ]:
!pip install wget
!apt-get -qq install -y curl
!pip install pygbif

import pandas as pd
import wget
import re
import tqdm
from urllib.parse import urlparse
from urllib.parse import parse_qs
from pygbif import occurrences as occ
import requests
from pygbif import species as gbifspecies
import hashlib
import numpy as np
import random
import os
import time
from shapely.geometry import shape, Polygon
import json
import subprocess
from google.colab import drive
drive.mount('/content/drive')

### Define WKT bounding box for London

In [ ]:
# Define bounding box for London derived from GLA boundaries

with open('/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/london.json') as f:
    geojson_data = json.load(f)

complex_poly = shape(geojson_data)
minx, miny, maxx, maxy = complex_poly.bounds
bounding_box_poly = Polygon([(minx, miny), (maxx, miny), (maxx, maxy), (minx, maxy), (minx, miny)])

print(bounding_box_poly.wkt)

POLYGON ((-0.510307110218469 51.28679028785072, 0.334043875762173 51.28679028785072, 0.334043875762173 51.691875617601774, -0.510307110218469 51.691875617601774, -0.510307110218469 51.28679028785072))


### Retrieve Occurrence Records from `taxa_final.csv` with Dual Search Criteria

Based on a CSV, this cell retrieves occurrence records for each taxon based on two `basisOfRecord` types: `HUMAN_OBSERVATION` and `PRESERVED_SPECIMEN` (with `lifeStage='Adult'`). It will attempt to match the the 'Number needed (Observed)' and 'Number needed (Preserved)' columns for each type of record.

This cell draws data from two datasets: iNaturalist and iRecord observations in London, and the Natural History Museum collections.

If those two datasets cannot meet the required number of observations, the supplementary search function in the following cell can augment the results to meet the "Number needed" requirements.

NB: Different rows have different "number needed" requirements for various reasons, eg:
- May be 0 for `HUMAN_OBSERVATION` if Insect Detect is going to provide 70% of the observations
- May be 6 for `PRESERVED_SPECIMEN` if the taxon to search for only makes up 20% of the required category (eg for some bumblebees). In that case, five taxa together would consitute 30 images or enough to train the AI model to recognise one category.

In [ ]:
# Load the 'taxa_final.csv' file into a pandas DataFrame named 'taxa_df'.
taxa_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/taxa_final.csv')

# Define a function to retrieve occurrence records from GBIF for the taxa in the DataFrame.
def get_occurrence_records_for_final_taxa(taxa_df, bounding_box_wkt, category_column='Category for GBIF download', name_column='Taxon (Latin name)'):

    # Initialize an empty dictionary to store all collected occurrence records, structured by category and taxon.
    all_occurrence_records = {}

    # Process the DataFrame and prepare to create a dictionary from it
    required_cols = [category_column, name_column, 'GBIF Taxon ID', 'Number needed (Observed)', 'Number needed (Preserved)']
    for col in required_cols:
        if col not in taxa_df.columns:
            print(f"ERROR: Missing required column '{col}' in the input DataFrame. Please ensure your CSV has this column.")
            return {}

    for index, row in tqdm.tqdm(taxa_df.iterrows(), total=len(taxa_df), desc="Retrieving all occurrence records"):
        category = row[category_column]
        taxon_name = row['Taxon (Latin name)']
        taxon_key = row['GBIF Taxon ID']
        num_needed_observed = row['Number needed (Observed)']
        num_needed_preserved = row['Number needed (Preserved)']

        # Extract 'Exception IDs' from the row. If not NaN, split the string by commas and convert to a set of integers.
        exception_ids_str = row.get('Exception IDs')
        exception_ids = set(map(int, exception_ids_str.split(','))) if pd.notna(exception_ids_str) else set()

        # Validate if the 'Taxon ID' is missing or not a valid number. If so, print an error and skip to the next row.
        if pd.isna(taxon_key):
            print(f"Skipping '{taxon_name}' (Category: {category}) due to missing Taxon ID.)")
            continue
        try:
            taxon_key = int(taxon_key)
        except ValueError:
            print(f"Skipping '{taxon_name}' (Category: {category}) due to invalid Taxon ID: {taxon_key} (Expected integer).")
            continue

        # Initialize the nested dictionary structure for the current category and taxon if they don't exist.
        if category not in all_occurrence_records:
            all_occurrence_records[category] = {}
        if taxon_name not in all_occurrence_records[category]:
            all_occurrence_records[category][taxon_name] = {'HUMAN_OBSERVATION': [], 'PRESERVED_SPECIMEN': []}

        # --- Search for HUMAN_OBSERVATION records ---

        if pd.notna(num_needed_observed) and num_needed_observed > 0:
            num_needed_observed = int(num_needed_observed) # Convert the number needed to an integer.

            print(f"\nRetrieving HUMAN_OBSERVATION records for category: {category}, taxon: {taxon_name} (needed: {num_needed_observed})")
            current_observed_records = []
            offset = 0 # Paginate API calls in case we need to sift through many records
            page_limit = 300

            # Loop until the target number of observed records is met.
            while len(current_observed_records) < num_needed_observed:
                remaining_to_fetch = num_needed_observed - len(current_observed_records)
                current_limit_gbif_call = min(page_limit, remaining_to_fetch)
                if current_limit_gbif_call <= 0: break

                try:
                    # Make an API call to GBIF to search for occurrence records.
                    # Remove lifeStage="Adult" from API call and apply programmatic filtering instead
                    results = occ.search(
                        taxonKey=taxon_key,
                        geometry=bounding_box_wkt,
                        basisOfRecord="HUMAN_OBSERVATION",
                        mediatype="StillImage",
                        occurrenceStatus="PRESENT",
                        limit=current_limit_gbif_call,
                        offset=offset,
                        fields="all",
                        datasetKey = [
                            "50c9509d-22c7-4a22-a47d-8c48425ef4a7", # iNaturalist
                            "51b7fe74-b4d9-4027-bd75-8425bfcf5300"  # iRecord
                        ]
                    )
                except Exception as e:
                    # Break the loop and print message if any errors during the GBIF API call.
                    print(f"ERROR: GBIF search failed for '{taxon_name}' (HUMAN_OBSERVATION). Error: {e}")
                    break

                # Get the list of results from the current page. If no results, break.
                current_page_records = results.get('results', [])
                if not current_page_records: break

                # Apply programmatic lifeStage filtering
                filtered_page_records = []
                for rec in current_page_records:
                    current_life_stage = rec.get('lifeStage')
                    # Only include records where lifeStage is 'Adult' or unspecified (None or empty string),
                    # and explicitly exclude records with lifeStage 'Larva'.
                    if (current_life_stage == 'Adult' or not current_life_stage) and current_life_stage != 'Larva':
                        filtered_page_records.append(rec)
                    else:
                        print(f"  Skipping HUMAN_OBSERVATION record {rec.get('key')} for {taxon_name} (lifeStage '{current_life_stage}' does not meet criteria).")

                # Process filtered records
                for rec in filtered_page_records:
                    if rec.get('key') in exception_ids:
                        print(f"  Skipping HUMAN_OBSERVATION record {rec.get('key')} for {taxon_name} (Exception ID).")
                        continue

                    # If the target number of records has been reached, stop adding.
                    if len(current_observed_records) >= num_needed_observed:
                        break

                    # Add the record to the list.
                    rec['search_stage'] = 'initial'
                    current_observed_records.append(rec)

                print(f"  Fetched {len(current_page_records)} raw records, {len(filtered_page_records)} passed lifeStage filter, total collected: {len(current_observed_records)}")

                # End search if no more records, otherwise loop back.
                if results.get('endOfRecords'):
                    if len(current_observed_records) < num_needed_observed:
                        print(f"  WARNING: Reached end of GBIF records for {taxon_name} HUMAN_OBSERVATION, but target ({num_needed_observed}) was not met. Collected: {len(current_observed_records)}")
                    break
                offset += page_limit
                time.sleep(0.5)

            # Store the collected observed records in the main dictionary.
            all_occurrence_records[category][taxon_name]['HUMAN_OBSERVATION'] = current_observed_records
            print(f"  Collected {len(current_observed_records)} HUMAN_OBSERVATION records for {taxon_name}.")

        # --- Search for PRESERVED_SPECIMEN records ---

        # Check if a number of preserved records is needed and is greater than 0.

        if pd.notna(num_needed_preserved) and num_needed_preserved > 0:
            num_needed_preserved = int(num_needed_preserved)

            print(f"\nRetrieving PRESERVED_SPECIMEN records for category: {category}, taxon: {taxon_name} (needed: {num_needed_preserved})")
            current_preserved_records = []
            offset = 0
            page_limit = 70

            # Loop until the target number of preserved records is met.
            while len(current_preserved_records) < num_needed_preserved:
                remaining_to_fetch = num_needed_preserved - len(current_preserved_records)
                current_limit_gbif_call = min(page_limit, remaining_to_fetch)
                if current_limit_gbif_call <= 0: break

                try:
                    # Make an API call to GBIF to search for occurrence records.
                    results = occ.search(
                        taxonKey=taxon_key,
                        basisOfRecord="PRESERVED_SPECIMEN",
                        mediatype="StillImage",
                        occurrenceStatus="PRESENT",
                        lifeStage="Adult", # Retain API-level lifeStage filter for PRESERVED_SPECIMEN as previously agreed
                        limit=current_limit_gbif_call,
                        offset=offset,
                        fields="all",
                        datasetKey = "7e380070-f762-11e1-a439-00145eb45e9a" # Natural History Museum
                    )
                except Exception as e:
                    print(f"ERROR: GBIF search failed for '{taxon_name}' (PRESERVED_SPECIMEN). Error: {e}")
                    break

                # Get the list of results from the current page. If no results, break.
                current_page_records = results.get('results', [])
                if not current_page_records: break

                # Skip each record if it matches an Exception ID.
                for rec in current_page_records:
                    if rec.get('key') in exception_ids:
                        print(f"  Skipping PRESERVED_SPECIMEN record {rec.get('key')} for {taxon_name} (Exception ID).")
                        continue

                    # If the target number of records has been reached, stop adding.
                    if len(current_preserved_records) >= num_needed_preserved:
                        break

                    # Add the record to the list.
                    rec['search_stage'] = 'initial'
                    current_preserved_records.append(rec)

                print(f"  Fetched {len(current_page_records)} raw records, total collected: {len(current_preserved_records)}")

                # End search if no more records, otherwise loop back.
                if results.get('endOfRecords'):
                    if len(current_preserved_records) < num_needed_preserved:
                        print(f"  WARNING: Reached end of GBIF records for {taxon_name} PRESERVED_SPECIMEN, but target ({num_needed_preserved}) was not met. Collected: {len(current_preserved_records)}")
                    break
                offset += page_limit
                time.sleep(0.5)

            # Store the collected preserved records in the main dictionary.
            all_occurrence_records[category][taxon_name]['PRESERVED_SPECIMEN'] = current_preserved_records
            print(f"  Collected {len(current_preserved_records)} PRESERVED_SPECIMEN records for {taxon_name}.")

    return all_occurrence_records

# Call the function to retrieve occurrence records for the selected taxa in the csv.

all_final_taxa_occurrence_records = get_occurrence_records_for_final_taxa(
    taxa_df,
    bounding_box_poly.wkt,
    category_column='Category for AI training',
    name_column='Taxon (Latin name)'
)

# Display a summary of the retrieved records.

print("\n--- Summary of all final taxa records ---")
for category, taxa_data in all_final_taxa_occurrence_records.items():
    print(f"Category: {category}")
    for taxon_name, records_by_type in taxa_data.items():
        observed_count = len(records_by_type.get('HUMAN_OBSERVATION', []))
        preserved_count = len(records_by_type.get('PRESERVED_SPECIMEN', []))
        print(f"  Taxon '{taxon_name}': {observed_count} observed records in iNaturalist in London, {preserved_count} preserved records in NHM collections")

### Secondary Search for Occurrence Records with Relaxed Constraints

This function will supplement the previously collected occurrence records if the initial searches did not meet the specified 'Number needed' for either `HUMAN_OBSERVATION` or `PRESERVED_SPECIMEN` types. It uses broader search criteria (UK-wide for iNaturalist and iRecord, no specific datasetKey for preserved specimens) to gather more data.

In [ ]:
# Define a function to supplement existing GBIF occurrence records with broader search parameters if targets were not met.

def supplement_occurrence_records_for_final_taxa(
    all_records_dict,
    taxa_df,
    category_column='Category for GBIF download',
    name_column='Taxon (Latin name)'
):

    print("\n--- Starting Secondary Search for Missing Records ---")

    # Extract information from the DataFrame
    for index, row in tqdm.tqdm(taxa_df.iterrows(), total=len(taxa_df), desc="Supplementing occurrence records"):
        category = row[category_column]
        taxon_name = row[name_column]
        taxon_key = row['GBIF Taxon ID']
        num_needed_observed = row['Number needed (Observed)']
        num_needed_preserved = row['Number needed (Preserved)']

        exception_ids_str = row.get('Exception IDs')
        exception_ids = set(map(int, exception_ids_str.split(','))) if pd.notna(exception_ids_str) else set()

        if pd.isna(taxon_key):
            continue
        try:
            taxon_key = int(taxon_key)
        except ValueError:
            continue

        # Ensure the nested dictionary structure exists for the current category and taxon.
        if category not in all_records_dict:
            all_records_dict[category] = {}
        if taxon_name not in all_records_dict[category]:
            all_records_dict[category][taxon_name] = {'HUMAN_OBSERVATION': [], 'PRESERVED_SPECIMEN': []}

        # Get references to the existing observed and preserved records lists.
        existing_observed_records = all_records_dict[category][taxon_name]['HUMAN_OBSERVATION']
        existing_preserved_records = all_records_dict[category][taxon_name]['PRESERVED_SPECIMEN']

        # Create sets of existing record keys to easily check for duplicates.
        existing_observed_ids = {r['key'] for r in existing_observed_records}
        existing_preserved_ids = {r['key'] for r in existing_preserved_records}

        # --- Supplement HUMAN_OBSERVATION records ---

        # Check if observed records are needed and the target is not yet met.
        if pd.notna(num_needed_observed) and num_needed_observed > 0:
            target_num_observed = int(num_needed_observed)
            currently_observed = len(existing_observed_records)

            if currently_observed < target_num_observed:
                print(f"\n  Supplementing HUMAN_OBSERVATION records for category: {category}, taxon: {taxon_name} (target: {target_num_observed}, current: {currently_observed})")
                offset = 0
                page_limit = 70

                # Loop to fetch more records until the target is met.
                while len(existing_observed_records) < target_num_observed:
                    remaining_to_fetch = target_num_observed - len(existing_observed_records)
                    current_limit_gbif_call = min(page_limit, remaining_to_fetch)
                    if current_limit_gbif_call <= 0: break

                    try:
                        # Make a broader GBIF API search for observed records
                        # Does not have to be iNaturalist
                        # Can be anywhere in GB, not just London
                        # Remove need for "Adult" life stage - instead check results

                        results = occ.search(
                            taxonKey=taxon_key,
                            country ="GB",
                            basisOfRecord="HUMAN_OBSERVATION",
                            mediatype="StillImage",
                            occurrenceStatus="PRESENT",
                            limit=current_limit_gbif_call,
                            offset=offset,
                            fields="all",
                            datasetKey = [
                            "50c9509d-22c7-4a22-a47d-8c48425ef4a7", # iNaturalist
                            "51b7fe74-b4d9-4027-bd75-8425bfcf5300"  # iRecord
                        ]
                        )
                    except Exception as e:
                        print(f"    ERROR: GBIF secondary search failed for '{taxon_name}' (HUMAN_OBSERVATION). Error: {e}")
                        break

                    current_page_records = results.get('results', [])
                    if not current_page_records: break

                    new_records_added_from_page = 0
                    for rec in current_page_records:
                        current_life_stage = rec.get('lifeStage')
                        # Only include records where lifeStage is 'Adult' or unspecified (None or empty string),
                        # and explicitly exclude 'Larva'.
                        if not ((current_life_stage == 'Adult' or not current_life_stage) and current_life_stage != 'Larva'):
                            print(f"    Skipping HUMAN_OBSERVATION record {rec.get('key')} for {taxon_name} (lifeStage '{current_life_stage}' does not meet criteria).")
                            continue

                        # Check for exception IDs
                        if rec.get('key') in exception_ids:
                            print(f"    Skipping HUMAN_OBSERVATION record {rec.get('key')} for {taxon_name} (Exception ID).")
                            continue

                        if len(existing_observed_records) >= target_num_observed: # Check before adding each record
                            break # Stop adding if target is reached

                        if rec['key'] not in existing_observed_ids:
                            rec['search_stage'] = 'supplementary'
                            existing_observed_records.append(rec)
                            existing_observed_ids.add(rec['key'])
                            new_records_added_from_page += 1

                    print(f"    Fetched {len(current_page_records)} raw records, added {new_records_added_from_page} unique observed records. Total for taxon: {len(existing_observed_records)}")

                    if results.get('endOfRecords'):
                        if len(existing_observed_records) < target_num_observed:
                            print(f"    WARNING: Reached end of GBIF records for {taxon_name} HUMAN_OBSERVATION, but target ({target_num_observed}) was not met. Collected: {len(existing_observed_records)}")
                        break
                    offset += page_limit
                    time.sleep(0.5)

        # --- Supplement PRESERVED_SPECIMEN records ---

        if pd.notna(num_needed_preserved) and num_needed_preserved > 0:
            target_num_preserved = int(num_needed_preserved)
            currently_preserved = len(existing_preserved_records)

            if currently_preserved < target_num_preserved:
                print(f"\n  Supplementing PRESERVED_SPECIMEN records for category: {category}, taxon: {taxon_name} (target: {target_num_preserved}, current: {currently_preserved})")
                offset = 0
                page_limit = 70

                while len(existing_preserved_records) < target_num_preserved:
                    remaining_to_fetch = target_num_preserved - len(existing_preserved_records)
                    current_limit_gbif_call = min(page_limit, remaining_to_fetch)
                    if current_limit_gbif_call <= 0: break # Safety break if target somehow already met

                    try:
                        # Make a broader GBIF API search for preserved records
                        # Does not have to be NHM
                        # Remove lifeStage filter from API call for programmatic filtering
                        results = occ.search(
                            taxonKey=taxon_key,
                            basisOfRecord="PRESERVED_SPECIMEN",
                            mediatype="StillImage",
                            occurrenceStatus="PRESENT",
                            limit=current_limit_gbif_call,
                            offset=offset,
                            fields="all"
                        )
                    except Exception as e:
                        print(f"    ERROR: GBIF secondary search failed for '{taxon_name}' (PRESERVED_SPECIMEN). Error: {e}")
                        break

                    current_page_records = results.get('results', [])
                    if not current_page_records: break

                    new_records_added_from_page = 0
                    for rec in current_page_records:
                        current_life_stage = rec.get('lifeStage')
                        # Only include records where lifeStage is 'Adult' or unspecified (None or empty string),
                        # and explicitly exclude 'Larva'.
                        if not ((current_life_stage == 'Adult' or not current_life_stage) and current_life_stage != 'Larva'):
                            print(f"    Skipping PRESERVED_SPECIMEN record {rec.get('key')} for {taxon_name} (lifeStage '{current_life_stage}' does not meet criteria).")
                            continue

                        # Check for exception IDs
                        if rec.get('key') in exception_ids:
                            print(f"    Skipping PRESERVED_SPECIMEN record {rec.get('key')} for {taxon_name} (Exception ID).")
                            continue

                        if len(existing_preserved_records) >= target_num_preserved: # Check before adding each record
                            break # Stop adding if target is reached
                        if rec['key'] not in existing_preserved_ids:
                            rec['search_stage'] = 'supplementary'
                            existing_preserved_records.append(rec)
                            existing_preserved_ids.add(rec['key'])
                            new_records_added_from_page += 1

                    print(f"    Fetched {len(current_page_records)} raw records, added {new_records_added_from_page} unique preserved records. Total for taxon: {len(existing_preserved_records)}")

                    if results.get('endOfRecords'):
                        if len(existing_preserved_records) < target_num_preserved:
                            print(f"    WARNING: Reached end of GBIF records for {taxon_name} PRESERVED_SPECIMEN, but target ({target_num_preserved}) was not met. Collected: {len(existing_preserved_records)}")
                        break
                    offset += page_limit
                    time.sleep(0.5)

    return all_records_dict

# Call the new function to supplement records
all_final_taxa_occurrence_records = supplement_occurrence_records_for_final_taxa(
    all_final_taxa_occurrence_records,
    taxa_df,
    category_column='Category for AI training',
    name_column='Taxon (Latin name)'
)

# Display summary of all combined records
print("\n--- Summary of ALL (initial + supplemented) final taxa records ---")
for category, taxa_data in all_final_taxa_occurrence_records.items():
    print(f"Category: {category}")
    for taxon_name, records_by_type in taxa_data.items():
        observed_count = len(records_by_type.get('HUMAN_OBSERVATION', []))
        preserved_count = len(records_by_type.get('PRESERVED_SPECIMEN', []))
        print(f"  Taxon '{taxon_name}': {observed_count} observed records, {preserved_count} preserved records")

### Final small extra search for rare species

Small scissor bees and sweat bees did not have enough human observations. This cell supplements these with worldwide observations. Try DE, CA, US as "country" in the occ.search area to get the final few images.

In [ ]:
# Tertiary search for two species which didn't have enough records in the UK

def extra_supplement_occurrence_records_for_final_taxa(
    all_records_dict,
    taxa_df,
    category_column='Category for GBIF download',
    name_column='Taxon (Latin name)'
):

    print("\n--- Starting Tertiary Search for Missing Records ---")

    # Extract information from the DataFrame
    for index, row in tqdm.tqdm(taxa_df.iterrows(), total=len(taxa_df), desc="Supplementing occurrence records"):
        category = row[category_column]
        taxon_name = row[name_column]
        taxon_key = row['GBIF Taxon ID']
        num_needed_observed = row['Number needed (Observed)']
        num_needed_preserved = row['Number needed (Preserved)']

        exception_ids_str = row.get('Exception IDs')
        exception_ids = set(map(int, exception_ids_str.split(','))) if pd.notna(exception_ids_str) else set()

        if pd.isna(taxon_key):
            continue
        try:
            taxon_key = int(taxon_key)
        except ValueError:
            continue

        # Ensure the nested dictionary structure exists for the current category and taxon.
        if category not in all_records_dict:
            all_records_dict[category] = {}
        if taxon_name not in all_records_dict[category]:
            all_records_dict[category][taxon_name] = {'HUMAN_OBSERVATION': [], 'PRESERVED_SPECIMEN': []}

        # Get references to the existing observed and preserved records lists.
        existing_observed_records = all_records_dict[category][taxon_name]['HUMAN_OBSERVATION']
        existing_preserved_records = all_records_dict[category][taxon_name]['PRESERVED_SPECIMEN']

        # Create sets of existing record keys to easily check for duplicates.
        existing_observed_ids = {r['key'] for r in existing_observed_records}
        existing_preserved_ids = {r['key'] for r in existing_preserved_records}

        # --- Supplement HUMAN_OBSERVATION records ---

        # Check if observed records are needed and the target is not yet met.
        if pd.notna(num_needed_observed) and num_needed_observed > 0:
            target_num_observed = int(num_needed_observed)
            currently_observed = len(existing_observed_records)

            if currently_observed < target_num_observed:
                print(f"\n  Supplementing HUMAN_OBSERVATION records for category: {category}, taxon: {taxon_name} (target: {target_num_observed}, current: {currently_observed})")
                offset = 0
                page_limit = 70

                # Loop to fetch more records until the target is met.
                while len(existing_observed_records) < target_num_observed:
                    remaining_to_fetch = target_num_observed - len(existing_observed_records)
                    current_limit_gbif_call = min(page_limit, remaining_to_fetch)
                    if current_limit_gbif_call <= 0: break

                    try:
                        # Remove country specification

                        results = occ.search(
                            taxonKey=taxon_key,
                            country=["US"],
                            basisOfRecord="HUMAN_OBSERVATION",
                            mediatype="StillImage",
                            occurrenceStatus="PRESENT",
                            limit=current_limit_gbif_call,
                            offset=offset,
                            fields="all",
                            datasetKey = [
                            "50c9509d-22c7-4a22-a47d-8c48425ef4a7", # iNaturalist
                            "51b7fe74-b4d9-4027-bd75-8425bfcf5300"  # iRecord
                        ]
                        )
                    except Exception as e:
                        print(f"    ERROR: GBIF secondary search failed for '{taxon_name}' (HUMAN_OBSERVATION). Error: {e}")
                        break

                    current_page_records = results.get('results', [])
                    if not current_page_records: break

                    new_records_added_from_page = 0
                    for rec in current_page_records:
                        current_life_stage = rec.get('lifeStage')
                        # Only include records where lifeStage is 'Adult' or unspecified (None or empty string),
                        # and explicitly exclude 'Larva'.
                        if not ((current_life_stage == 'Adult' or not current_life_stage) and current_life_stage != 'Larva'):
                            print(f"    Skipping HUMAN_OBSERVATION record {rec.get('key')} for {taxon_name} (lifeStage '{current_life_stage}' does not meet criteria).")
                            continue

                        # Check for exception IDs
                        if rec.get('key') in exception_ids:
                            print(f"    Skipping HUMAN_OBSERVATION record {rec.get('key')} for {taxon_name} (Exception ID).")
                            continue

                        if len(existing_observed_records) >= target_num_observed: # Check before adding each record
                            break # Stop adding if target is reached

                        if rec['key'] not in existing_observed_ids:
                            rec['search_stage'] = 'supplementary'
                            existing_observed_records.append(rec)
                            existing_observed_ids.add(rec['key'])
                            new_records_added_from_page += 1

                    print(f"    Fetched {len(current_page_records)} raw records, added {new_records_added_from_page} unique observed records. Total for taxon: {len(existing_observed_records)}")

                    if results.get('endOfRecords'):
                        if len(existing_observed_records) < target_num_observed:
                            print(f"    WARNING: Reached end of GBIF records for {taxon_name} HUMAN_OBSERVATION, but target ({target_num_observed}) was not met. Collected: {len(existing_observed_records)}")
                        break
                    offset += page_limit
                    time.sleep(0.5)

    return all_records_dict

# Call the new function to supplement records
all_final_taxa_occurrence_records = extra_supplement_occurrence_records_for_final_taxa(
    all_final_taxa_occurrence_records,
    taxa_df,
    category_column='Category for AI training',
    name_column='Taxon (Latin name)'
)

# Display summary of all combined records
print("\n--- Summary of ALL (initial + supplemented) final taxa records ---")
for category, taxa_data in all_final_taxa_occurrence_records.items():
    print(f"Category: {category}")
    for taxon_name, records_by_type in taxa_data.items():
        observed_count = len(records_by_type.get('HUMAN_OBSERVATION', []))
        preserved_count = len(records_by_type.get('PRESERVED_SPECIMEN', []))
        print(f"  Taxon '{taxon_name}': {observed_count} observed records, {preserved_count} preserved records")

### Save Collected Occurrence Records to CSV

Flattens the `all_final_taxa_occurrence_records` dictionary into a pandas DataFrame and save it as a CSV file in Google Drive.

In [ ]:
flattened_records = []

for category, taxa_data in all_final_taxa_occurrence_records.items():
    for taxon_name, records_by_type in taxa_data.items():
        for record_type, records_list in records_by_type.items():
            for rec in records_list:
                record_details = {
                    'Category': category,
                    'Taxon (Latin name)': taxon_name,
                    'Basis of Record': record_type,
                    'Occurrence ID': rec.get('key'),
                    'Search Stage': rec.get('search_stage', 'unknown')
                }
                if 'gbifID' in rec: record_details['gbifID'] = rec['gbifID']
                if 'scientificName' in rec: record_details['scientificName'] = rec['scientificName']
                if 'eventDate' in rec: record_details['eventDate'] = rec['eventDate']
                if 'decimalLatitude' in rec: record_details['decimalLatitude'] = rec['decimalLatitude']
                if 'decimalLongitude' in rec: record_details['decimalLongitude'] = rec['decimalLongitude']
                if 'datasetKey' in rec: record_details['datasetKey'] = rec['datasetKey']
                if 'media' in rec and len(rec['media']) > 0: record_details['media_url'] = rec['media'][0].get('identifier')
                if 'lifeStage' in rec: record_details['lifeStage'] = rec['lifeStage']

                flattened_records.append(record_details)

output_df = pd.DataFrame(flattened_records)
output_csv_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/gbif_occurrence_records.csv'
output_df.to_csv(output_csv_path, index=False)


### Regenerate `all_final_taxa_occurrence_records` from CSV

In [ ]:
import pandas as pd
import json

# Define the path to the CSV file
output_csv_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/gbif_occurrence_records.csv'

# Load the CSV file into a DataFrame
reconstructed_df = pd.read_csv(output_csv_path)

# Initialize an empty dictionary to store the reconstructed records
reconstructed_all_occurrence_records = {}

# Iterate through the DataFrame rows and rebuild the nested dictionary structure
for index, row in reconstructed_df.iterrows():
    category = row['Category']
    taxon_name = row['Taxon (Latin name)']
    record_type = row['Basis of Record']

    # Ensure the nested dictionary structure exists
    if category not in reconstructed_all_occurrence_records:
        reconstructed_all_occurrence_records[category] = {}
    if taxon_name not in reconstructed_all_occurrence_records[category]:
        reconstructed_all_occurrence_records[category][taxon_name] = {'HUMAN_OBSERVATION': [], 'PRESERVED_SPECIMEN': []}

    # Create a record dictionary from the row, ensuring correct types for some fields
    record = {
        'key': row['Occurrence ID'],
        'search_stage': row['Search Stage']
    }

    # Add other fields if they exist in the row and are not NaN
    if pd.notna(row.get('gbifID')): record['gbifID'] = row['gbifID']
    if pd.notna(row.get('scientificName')): record['scientificName'] = row['scientificName']
    if pd.notna(row.get('eventDate')): record['eventDate'] = row['eventDate']
    if pd.notna(row.get('decimalLatitude')): record['decimalLatitude'] = row['decimalLatitude']
    if pd.notna(row.get('decimalLongitude')): record['decimalLongitude'] = row['decimalLongitude']
    if pd.notna(row.get('datasetKey')): record['datasetKey'] = row['datasetKey']
    if pd.notna(row.get('media_url')): record['media'] = [{'identifier': row['media_url']}] # Recreate media list format
    if pd.notna(row.get('lifeStage')): record['lifeStage'] = row['lifeStage']

    reconstructed_all_occurrence_records[category][taxon_name][record_type].append(record)

# Assign the reconstructed dictionary to all_final_taxa_occurrence_records
all_final_taxa_occurrence_records = reconstructed_all_occurrence_records

# Display a summary of the reconstructed records to confirm
print("\n--- Summary of RECONSTRUCTED final taxa records from CSV ---")
for category, taxa_data in all_final_taxa_occurrence_records.items():
    print(f"Category: {category}")
    for taxon_name, records_by_type in taxa_data.items():
        observed_count = len(records_by_type.get('HUMAN_OBSERVATION', []))
        preserved_count = len(records_by_type.get('PRESERVED_SPECIMEN', []))
        print(f"  Taxon '{taxon_name}': {observed_count} observed records, {preserved_count} preserved records")


### Define functions for downloading images using hashlib

In [ ]:
# Define functions to get GBIF links with hash method

def get_gbif_link(gbifid, hashnumber):
    part = 'https://www.gbif.org/tools/zoom/simple.html?src=//api.gbif.org/v1/image/cache/occurrence/'
    return part + str(gbifid) + '/media/' + str(hashnumber)

def generate_gbif_hash_url(gbif_id, original_identifier_url):

    if not original_identifier_url:
        return None
    try:
        # Calculate the MD5 hash from the original identifier URL
        hashnumber = hashlib.md5(original_identifier_url.encode("utf-8")).hexdigest()
        # Use the get_gbif_link function to construct the hash-based URL
        return get_gbif_link(gbif_id, hashnumber)
    except Exception as e:
        print(f"Error generating hash-based URL for GBIF ID {gbif_id} with URL {original_identifier_url}: {e}")
        return None

# Define a function to download an image from a given URL.

def download_image(url, name, pth_to_save, save_format='jpg'):
    final_image_url = url

    # Check if the URL is a GBIF redirect URL that needs to be parsed to find the actual image source.
    if "gbif.org/tools/zoom" in url:
        try:
            parsed_url = urlparse(url)
            q = parse_qs(parsed_url.query)
            if 'src' in q and q['src']:
                src_url = q['src'][0]
                if src_url.startswith('//'):
                    final_image_url = f"https:{src_url}" # If protocol-relative, prepend 'https:'.
                else:
                    final_image_url = src_url # Otherwise, use the URL as is.
            else:

                print(f'ERROR in download_image (parsing GBIF redirect): No "src" parameter found in query for URL: {url}')
                return
        except KeyError:

            print(f'ERROR in download_image (parsing GBIF redirect): KeyError accessing "src" for URL: {url}')
            return
        except Exception as e:

            print(f'ERROR in download_image (parsing GBIF redirect): {type(e).__name__} - {e} for URL: {url}')
            return

    file_path = os.path.join(pth_to_save, f"{name}.{save_format}")

    try:
        # Use the 'curl' command-line tool for downloading files.
        # '-L' tells curl to follow HTTP redirects.
        # '-sS' makes curl operate in silent mode but still show errors.
        # '-o' specifies the output file name.
        # '-A' sets a custom User-Agent header, which can help bypass some server restrictions.
        command = [
            "curl",
            "-L",
            "-sS",
            "-o",
            file_path,
            "-A",
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
            final_image_url
        ]
        # Execute the curl command using subprocess.run.
        # 'capture_output=True' captures stdout and stderr.
        # 'text=True' decodes stdout/stderr as text.
        # 'check=True' raises a CalledProcessError if the command returns a non-zero exit code.
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        # if result.stderr: # curl -S will print errors to stderr
        #     print(f"ERROR from curl for {name}: {result.stderr.strip()}") # This line is commented out, so curl errors are not explicitly printed here unless 'check=True' fails.

    except subprocess.CalledProcessError as e:
        # Catch errors specifically from the subprocess call (e.g., non-zero exit code from curl).
        print(f"ERROR: Failed to download image {name} from {final_image_url} using curl. Exit code: {e.returncode}, Error: {e.stderr.strip()}")
    except Exception as e:
        # Catch any other unexpected errors during the download process.
        print(f"ERROR: An unexpected error occurred while downloading {name} from {final_image_url} using curl: {type(e).__name__} - {e}")

### Download images using hashlib

In [ ]:
image_save_base_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images'

# Check if the base directory for saving images already exists. If it doesn't exist, create it.
if not os.path.exists(image_save_base_path):
    os.makedirs(image_save_base_path)
    print(f"Created directory: {image_save_base_path}")
else:
    print(f"Directory already exists: {image_save_base_path}")

downloaded_image_count = 0
skipped_image_count = 0

# Iterate through the 'all_final_taxa_occurrence_records' dictionary to process and download images.
for category, taxa_data in tqdm.tqdm(all_final_taxa_occurrence_records.items(), desc="Downloading images"):
    category_safe = re.sub(r'[^a-zA-Z0-9_]', '_', category)   # Sanitize the category name for use in filenames.
    category_path = os.path.join(image_save_base_path, category_safe)

    if not os.path.exists(category_path):
        os.makedirs(category_path)


    for taxon_name, records_by_type in taxa_data.items():
        taxon_name_safe = re.sub(r'[^a-zA-Z0-9_]', '_', taxon_name)

        # Iterate through each record type (e.g., 'HUMAN_OBSERVATION', 'PRESERVED_SPECIMEN').
        for record_type, records_list in records_by_type.items():
            # Create a subfolder for each record_type (e.g., 'HUMAN_OBSERVATION', 'PRESERVED_SPECIMEN')
            record_type_path = os.path.join(category_path, record_type)
            if not os.path.exists(record_type_path):
                os.makedirs(record_type_path)

            for rec in records_list:
                image_url = None
                gbif_occurrence_id = rec.get('key')
                # Check if 'media' field exists in the record and has at least one entry.
                if 'media' in rec and len(rec['media']) > 0:
                    image_url = rec['media'][0].get('identifier') # Get the URL of the first image.

                if image_url and gbif_occurrence_id:
                    search_stage = rec.get('search_stage', 'unknown_stage')

                    # Create a unique filename for the image using category, taxon, occurrence ID, and search stage.
                    # Changed separator from '_' to '__'
                    image_name = f"{category_safe}__{taxon_name_safe}__{gbif_occurrence_id}__{search_stage}"
                    file_path = os.path.join(record_type_path, f"{image_name}.jpg") # Assuming JPG format

                    # Check if the file already exists, if so, skip downloading
                    if os.path.exists(file_path):
                        # print(f"  Skipping existing image: {image_name}.jpg") # Uncomment for verbose skipping
                        skipped_image_count += 1
                        continue

                    # Attempt to generate a hash-based URL
                    download_url = generate_gbif_hash_url(gbif_occurrence_id, image_url)

                    # Fallback to original image_url if hash-based URL generation fails or returns None
                    if not download_url:
                        download_url = image_url

                    try:
                        # Call the 'download_image' function to download the image into the record_type_path.
                        download_image(download_url, image_name, record_type_path)
                        downloaded_image_count += 1
                    except Exception as e:
                        print(f"  Error downloading image {image_name} from {download_url}: {e}")

# After all records have been processed, print the total number of images downloaded.
print(f"\nSuccessfully downloaded {downloaded_image_count} new images to {image_save_base_path}")
print(f"Skipped {skipped_image_count} existing images.")

Directory already exists: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images



Successfully downloaded 0 new images to /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images
Skipped 0 existing images.


### Count images in each category folder to confirm correct numbers

In [ ]:
print(f"\nCounting images in each subfolder within: {image_save_base_path}")

category_image_counts = {}

for category_folder_name in os.listdir(image_save_base_path):
    category_path = os.path.join(image_save_base_path, category_folder_name)

    if os.path.isdir(category_path):
        # Initialize counts for this category
        category_image_counts[category_folder_name] = {'HUMAN_OBSERVATION': 0, 'PRESERVED_SPECIMEN': 0, 'TOTAL': 0}

        # Iterate through the record type subfolders
        for record_type_folder_name in os.listdir(category_path):
            record_type_path = os.path.join(category_path, record_type_folder_name)

            if os.path.isdir(record_type_path) and record_type_folder_name in ['HUMAN_OBSERVATION', 'PRESERVED_SPECIMEN']:
                image_files = [f for f in os.listdir(record_type_path) if f.lower().endswith('.jpg')]
                count = len(image_files)
                category_image_counts[category_folder_name][record_type_folder_name] = count
                category_image_counts[category_folder_name]['TOTAL'] += count

# Display the counts
if category_image_counts:
    for category, counts_by_type in category_image_counts.items():
        print(f"  Category '{category}':")
        print(f"    - HUMAN_OBSERVATION: {counts_by_type['HUMAN_OBSERVATION']} images")
        print(f"    - PRESERVED_SPECIMEN: {counts_by_type['PRESERVED_SPECIMEN']} images")
        print(f"    - Total for category: {counts_by_type['TOTAL']} images")
else:
    print("No image categories found or no images downloaded.")


Counting images in each subfolder within: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images
  Category 'Other_fly':
    - HUMAN_OBSERVATION: 0 images
    - PRESERVED_SPECIMEN: 30 images
    - Total for category: 30 images
  Category 'Drone_fly':
    - HUMAN_OBSERVATION: 0 images
    - PRESERVED_SPECIMEN: 30 images
    - Total for category: 30 images
  Category 'Batman_hoverfly':
    - HUMAN_OBSERVATION: 0 images
    - PRESERVED_SPECIMEN: 30 images
    - Total for category: 30 images
  Category 'Globetail_hoverfly':
    - HUMAN_OBSERVATION: 0 images
    - PRESERVED_SPECIMEN: 31 images
    - Total for category: 31 images
  Category 'Humming_Syrphus':
    - HUMAN_OBSERVATION: 0 images
    - PRESERVED_SPECIMEN: 30 images
    - Total for category: 30 images
  Category 'Marmalade_hoverfly':
    - HUMAN_OBSERVATION: 0 images
    - PRESERVED_SPECIMEN: 30 images
    - Total for category: 30 images
  Category 'Narcissus_bulb_fly':
    - HUMAN_OBSERVATION: 70 images
  

### Delete 0-byte image files

In [ ]:
deleted_count = 0

print(f"Scanning '{image_save_base_path}' for 0-byte image files...")

for root, dirs, files in os.walk(image_save_base_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
            file_path = os.path.join(root, file)
            try:
                if os.path.getsize(file_path) == 0:
                    os.remove(file_path)
                    deleted_count += 1
                    print(f"  Deleted 0-byte file: {file_path}")
            except OSError as e:
                print(f"  Error processing file {file_path}: {e}")

print(f"\nFinished deleting 0-byte files. Total deleted: {deleted_count}")

Scanning '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images' for 0-byte image files...

Finished deleting 0-byte files. Total deleted: 0


### Remove 0-byte image occurrences and re-supplement

In [ ]:
import re # Ensure re is imported for parsing
import os # Ensure os is imported for path manipulation

# Get the standard output from the cell that deleted 0-byte files (cell ID: b6a8fac8)
# This assumes the standard output is available from the execution context.
# In a real Colab environment, this would typically be retrieved from previous cell output.
# For the purpose of this example, I'll use the provided context output directly.

# NOTE: This 'stdout_from_delete_cell' content is hardcoded based on the last execution of cell b6a8fac8.
# If you run cell b6a8fac8 again and its output changes, this variable should be updated accordingly.
stdout_from_delete_cell = """
Scanning '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images' for 0-byte image files...
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Thick_legged_hoverfly/PRESERVED_SPECIMEN/Thick_legged_hoverfly__Syritta_pipiens__1837876358__initial.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Flower_bee/PRESERVED_SPECIMEN/Flower_bee__Anthophora__4849200421__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Flower_bee/PRESERVED_SPECIMEN/Flower_bee__Anthophora__4454116538__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838421585__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838422213__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838422271__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838422458__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838422880__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838423250__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838423274__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Yellow_faced_bee/PRESERVED_SPECIMEN/Yellow_faced_bee__Hylaeus__5838423290__supplementary.jpg
  Deleted 0-byte file: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images/Honeybee/PRESERVED_SPECIMEN/Honeybee__Apis_mellifera__4849200427__supplementary.jpg

Finished deleting 0-byte files. Total deleted: 12
"""

deleted_0byte_files_info = []
# Parse the stdout to get the list of deleted files
for line in stdout_from_delete_cell.splitlines():
    if line.strip().startswith("Deleted 0-byte file:"):
        file_path = line.split(":", 1)[1].strip()

        record_type = os.path.basename(os.path.dirname(file_path))
        filename_without_ext = os.path.splitext(os.path.basename(file_path))[0]
        parts = filename_without_ext.split('__')

        if len(parts) >= 3: # Expect at least category, taxon, occurrence ID
            category_safe_from_file = parts[0]
            taxon_name_safe_from_file = parts[1]
            occurrence_id_str = parts[2]
            deleted_0byte_files_info.append((category_safe_from_file, taxon_name_safe_from_file, record_type, occurrence_id_str))

print(f"Parsed {len(deleted_0byte_files_info)} occurrences with 0-byte images.\n")

if deleted_0byte_files_info:
    # Create a mapping from sanitized names back to original names for lookup
    sanitized_category_to_original = {}
    sanitized_taxon_to_original = {}

    # The sanitization logic should match the one used during image downloading
    # which was: re.sub(r'[^a-zA-Z0-9_]', '_', name)
    for index, row in taxa_df.iterrows():
        original_category = row['Category for AI training']
        original_taxon_name = row['Taxon (Latin name)']

        sanitized_category = re.sub(r'[^a-zA-Z0-9_]', '_', original_category)
        sanitized_taxon_name = re.sub(r'[^a-zA-Z0-9_]', '_', original_taxon_name)

        sanitized_category_to_original[sanitized_category] = original_category
        sanitized_taxon_to_original[sanitized_taxon_name] = original_taxon_name


    print("Removing 0-byte image records from all_final_taxa_occurrence_records...")
    removed_count = 0
    for category_safe_from_file, taxon_name_safe_from_file, record_type, occurrence_id_str in deleted_0byte_files_info:
        # Use the mappings to get the original category and taxon name
        category = sanitized_category_to_original.get(category_safe_from_file)
        taxon_name = sanitized_taxon_to_original.get(taxon_name_safe_from_file)

        if category is None or taxon_name is None:
            print(f"  Warning: Could not find original category/taxon for sanitized names: '{category_safe_from_file}', '{taxon_name_safe_from_file}'. Skipping record.")
            continue

        occurrence_id = int(occurrence_id_str)

        # Now use the correctly mapped original category and taxon_name for dictionary lookup
        if category in all_final_taxa_occurrence_records and taxon_name in all_final_taxa_occurrence_records[category]:
            records_list = all_final_taxa_occurrence_records[category][taxon_name].get(record_type, [])
            initial_len = len(records_list)
            all_final_taxa_occurrence_records[category][taxon_name][record_type] = [
                rec for rec in records_list if rec.get('key') != occurrence_id
            ]
            if len(all_final_taxa_occurrence_records[category][taxon_name][record_type]) < initial_len:
                removed_count += 1
                print(f"  Removed {record_type} record with ID {occurrence_id} for {taxon_name} in {category}")

    print(f"Finished removing {removed_count} records.\n")

    # Re-run the supplementation function to find replacement records
    print("Re-running supplementation to find new records...")
    all_final_taxa_occurrence_records = supplement_occurrence_records_for_final_taxa(
        all_final_taxa_occurrence_records,
        taxa_df,
        category_column='Category for AI training',
        name_column='Taxon (Latin name)'
    )

    # Display summary of all combined records after re-supplementation
    print("\n--- Summary of ALL (after 0-byte removal and re-supplementation) final taxa records ---")
    for category, taxa_data in all_final_taxa_occurrence_records.items():
        print(f"Category: {category}")
        for taxon_name, records_by_type in taxa_data.items():
            observed_count = len(records_by_type.get('HUMAN_OBSERVATION', []))
            preserved_count = len(records_by_type.get('PRESERVED_SPECIMEN', []))
            print(f"  Taxon '{taxon_name}': {observed_count} observed records, {preserved_count} preserved records")
else:
    print("No 0-byte image files were reported as deleted. No records will be removed or re-supplemented.")

Parsed 12 occurrences with 0-byte images.

Removing 0-byte image records from all_final_taxa_occurrence_records...
  Removed PRESERVED_SPECIMEN record with ID 5838421585 for Hylaeus in Yellow-faced bee
  Removed PRESERVED_SPECIMEN record with ID 5838422213 for Hylaeus in Yellow-faced bee
Finished removing 2 records.

Re-running supplementation to find new records...

--- Starting Secondary Search for Missing Records ---


Supplementing occurrence records:   0%|          | 0/39 [00:00<?, ?it/s]


  Supplementing PRESERVED_SPECIMEN records for category: Yellow-faced bee, taxon: Hylaeus (target: 30, current: 28)
    Fetched 2 raw records, added 0 unique preserved records. Total for taxon: 28
    Fetched 2 raw records, added 0 unique preserved records. Total for taxon: 28
    Fetched 2 raw records, added 0 unique preserved records. Total for taxon: 28
    Fetched 2 raw records, added 1 unique preserved records. Total for taxon: 29
    Fetched 1 raw records, added 1 unique preserved records. Total for taxon: 30


Supplementing occurrence records: 100%|██████████| 39/39 [00:04<00:00,  7.91it/s]


--- Summary of ALL (after 0-byte removal and re-supplementation) final taxa records ---
Category: Other fly
  Taxon 'Diptera except Syrphidae': 0 observed records, 30 preserved records
Category: Drone fly
  Taxon 'Eristalis': 0 observed records, 30 preserved records
Category: Batman hoverfly
  Taxon 'Myathropa florea': 0 observed records, 30 preserved records
Category: Globetail hoverfly
  Taxon 'Sphaerophoria': 0 observed records, 30 preserved records
Category: Humming Syrphus
  Taxon 'Syrphus': 0 observed records, 30 preserved records
Category: Marmalade hoverfly
  Taxon 'Episyrphus balteatus': 0 observed records, 30 preserved records
Category: Narcissus bulb fly
  Taxon 'Merodon equestris': 70 observed records, 30 preserved records
Category: Thick-legged hoverfly
  Taxon 'Syritta pipiens': 70 observed records, 30 preserved records
Category: Ant
  Taxon 'Formicidae': 0 observed records, 30 preserved records
Category: Flower bee
  Taxon 'Anthophora': 70 observed records, 30 preserved